In [9]:
import os
import random
import itertools 
from pathlib import Path

import pandas as pd
import numpy as np
from cblearn.datasets import make_all_triplet_indices


TRIAL_PER_BLOCK = 150
TRIAL_PER_TRAINING = 120
BLOCKS_PER_SESSION = 6

# Create triplet condition files

## Load objects of the three image sets

In [10]:
# folder that contains image folders, i.e. taxonomic_images, action_images, Greebles .
root_dir = Path("..")

set_names = ('action', 'greebles', 'taxonomic')
set_dfs = {}
for name in set_names:
    df = pd.read_csv(root_dir / f'resources/{name}_info.csv')
    df['image_set'] = name
    set_dfs[name] = df
    num_triplets = 2*2*len(df)*np.log2(len(df))
    print(f"{name}: {len(df)} images, {int(num_triplets)}-{2 * int(num_triplets)}  triplets")

action: 18 images, 300-600  triplets
greebles: 18 images, 300-600  triplets
taxonomic: 18 images, 300-600  triplets


In [11]:
# sample all triplets per dataset
rng = np.random.default_rng(15122022)
triplet_dfs = []
for name, set_df in set_dfs.items():
    triplets = make_all_triplet_indices(len(set_df), monotonic=False)
    rng.shuffle(triplets, axis=0)
    files = set_df['File_name'].values
    trial_per_block = TRIAL_PER_BLOCK // len(set_dfs)
    blocks = np.repeat(np.arange(np.ceil(len(triplets) / trial_per_block)), trial_per_block)[:len(triplets)].astype(int)
    triplet_df = pd.DataFrame(
        {'image1': files[triplets[:, 0]], 
        'middle': files[triplets[:, 1]], 
        'image2': files[triplets[:, 2]],
        'image1_index': triplets[:, 0],
        'middle_index': triplets[:, 1],
        'image2_index': triplets[:, 2],
        'image_set': name,
        'block_number': blocks + 1,
        'session_number': 0,
        'is_left': rng.choice([1, 2], size=len(triplets)),
        })
    triplet_dfs.append(triplet_df)
triplet_df = pd.concat(triplet_dfs)

# join triplets of all image sets
for block in triplet_df.block_number.unique():
    session = ((block - 1) // BLOCKS_PER_SESSION) + 1
    triplet_df.loc[triplet_df.block_number == block, 'session_number'] = session
print(len(triplet_df))

7344


In [12]:
# create training samples
training_triplet_df = triplet_df.groupby('image_set', group_keys=False).apply(lambda x: x.sample(n=TRIAL_PER_TRAINING//len(set_dfs), random_state=rng))  # sample stratified from image sets
training_triplet_df = training_triplet_df.sample(frac=1, random_state=rng)  # random order within blocks
training_triplet_df['block_number'] = 0
training_triplet_df['session_number'] = 0
cond_file = root_dir / f'experiment/conditions/training.csv'
print(training_triplet_df.columns)
if cond_file.exists():
    raise ValueError(f'File {cond_file} exists!')
training_triplet_df.to_csv(cond_file, index_label='triplet_index')


Index(['image1', 'middle', 'image2', 'image1_index', 'middle_index',
       'image2_index', 'image_set', 'block_number', 'session_number',
       'is_left'],
      dtype='object')


In [14]:
# write sessions with shuffled repetitions

rng = np.random.default_rng(24122022)
is_left = triplet_df['is_left'].values
for rep in 'abcdefghijklmnopqrst':
    this_triplet_df = triplet_df.groupby('block_number').sample(frac=1, random_state=rng)  # random order within blocks
    this_triplet_df['repetition'] = rep
    this_triplet_df['is_left'] = is_left  # alternate left/right
    for session, df in this_triplet_df.groupby('session_number'):
        cond_file = root_dir / f'experiment/conditions/triplets_3x18_session{session}{rep}.csv'
        if cond_file.exists():
            raise ValueError(f'File {cond_file} exists!')
        df.to_csv(cond_file, index_label='triplet_index')
    is_left = np.where(is_left == 1, 2, 1)  # flip image1-image2